# Speech Emotion Recognition on CREMA-D (Classical vs Quantum)

Main Colab-ready notebook for training/evaluating classical and hybrid quantum models on CREMA-D. Supports spectrogram CNNs, MFCC CNNs, and precomputed embeddings with optional two-stage fine-tuning.


In [ ]:
# Colab setup: mount Drive, clone repo, checkout fine-tunning branch
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/lburdman/qnn-transfer-learning.git'
REPO_PATH = Path('/content/qnn-transfer-learning')
BRANCH = 'fine-tunning'

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not REPO_PATH.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)], check=True)
    os.chdir(REPO_PATH)
    subprocess.run(['git', 'fetch'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
else:
    os.chdir(Path('.'))

sys.path.append(str(Path.cwd() / 'src'))
print('Working dir:', Path.cwd())
print('Python path updated with src/')


In [ ]:
import json, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from src.utils import configure_run
from src.dataset import create_dataloaders_all
from src.model_builder import build_model
from src.training import (
    train_with_history,
    train_model,
    evaluate_model,
    pretrain_backbone_and_embedding,
    finetune_head_only,
    summarize_experiments,
    freeze_module_params,
)
from src.quantum_circuit import draw_qnode_circuit_example, analyze_trained_quantum_head

print('Imports ready')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


In [ ]:
# Experiment configuration
DATA_ROOT = '/content/drive/MyDrive/CREMAD' if 'google.colab' in sys.modules else str(Path.cwd() / 'CREMAD')
BASE_MODEL = 'emb_resnet18'  # options: emb_resnet18, emb_vgg16, emb_panns_cnn14, cnn_specs, cnn_mfcc
USE_QUANTUM = False
FINE_TUNING = True  # True: new two-stage pipeline; False: legacy hybrid pipeline
SELECTED_CLASSES = None  # e.g., ['ANG', 'SAD'] or None for all
N_QUBITS = 10
Q_DEPTH = 2
BATCH_SIZE = 8
EPOCHS_PRETRAIN = 4
EPOCHS_FINETUNE = 8
LR_PRETRAIN = 1e-3
LR_FINETUNE = 5e-4
RNG_SEED = 42

random.seed(RNG_SEED); np.random.seed(RNG_SEED); torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)

config = configure_run(
    base_model=BASE_MODEL,
    quantum=USE_QUANTUM,
    classical_model='512_nq_2',
    n_qubits=N_QUBITS,
    q_depth=Q_DEPTH,
    selected_classes=SELECTED_CLASSES,
    batch_size=BATCH_SIZE,
    num_epochs=EPOCHS_FINETUNE,
    learning_rate=LR_FINETUNE,
    data_root=DATA_ROOT,
)
config['num_epochs_pretrain'] = EPOCHS_PRETRAIN
config['learning_rate_pretrain'] = LR_PRETRAIN
print(json.dumps(config, indent=2))


In [ ]:
dataloaders, dataset_sizes, class_names, counts_per_class = create_dataloaders_all(config, shuffle=True, num_workers=2)
print('Splits sizes:', dataset_sizes)
print('Classes:', class_names)
phase = 'train' if 'train' in dataloaders else list(dataloaders.keys())[0]
sample_x, sample_y = next(iter(dataloaders[phase]))
print('Sample batch shape:', sample_x.shape)
print('Label shape:', sample_y.shape)
print('Counts per class (train):', counts_per_class.get('train', {}))


In [ ]:
if USE_QUANTUM:
    try:
        draw_qnode_circuit_example(n_qubits=N_QUBITS, q_depth=Q_DEPTH, max_layers=None, seed=0)
    except Exception as exc:
        print('Quantum preview skipped:', exc)
else:
    print('Quantum preview skipped (USE_QUANTUM=False)')


In [ ]:
history_stage1 = {}
history_stage2 = {}
final_acc = None
if FINE_TUNING:
    if BASE_MODEL in ['cnn_specs', 'cnn_mfcc']:
        ft_cfg = config.copy()
        ft_cfg.update({
            'pretrain_epochs': EPOCHS_PRETRAIN,
            'finetune_epochs': EPOCHS_FINETUNE,
            'learning_rate_pretrain': LR_PRETRAIN,
            'learning_rate_finetune_classical': LR_FINETUNE,
            'learning_rate_finetune_quantum': LR_FINETUNE,
            'n_qubits': N_QUBITS,
            'q_depth': Q_DEPTH,
            'backbone_dir': str(Path(DATA_ROOT) / 'Models' / 'backbone'),
        })
        _, history_stage1, checkpoint = pretrain_backbone_and_embedding(dataloaders, dataset_sizes, class_names, ft_cfg, representation_tag=BASE_MODEL.split('_')[-1])
        head_type = 'quantum' if USE_QUANTUM else 'classical'
        _, history_stage2, final_acc = finetune_head_only(checkpoint, dataloaders, dataset_sizes, class_names, ft_cfg, head_type=head_type, representation_tag=BASE_MODEL.split('_')[-1])
        eval_model = _
    else:
        input_dim = sample_x.shape[1] if sample_x.ndim == 2 else sample_x.shape[1]
        feature_mapper = nn.Sequential(
            nn.Linear(input_dim, max(64, N_QUBITS * 2)),
            nn.ReLU(),
            nn.Linear(max(64, N_QUBITS * 2), N_QUBITS),
            nn.ReLU(),
        ).to(device)
        head_stage1 = nn.Linear(N_QUBITS, len(class_names)).to(device)
        stage1_model = nn.Sequential(feature_mapper, head_stage1)
        model_dir_stage1 = Path(config['model_dir']) / 'stage1'
        model_dir_stage1.mkdir(parents=True, exist_ok=True)
        stage1_model, history_stage1 = train_with_history(stage1_model, dataloaders, dataset_sizes, device=device, num_epochs=EPOCHS_PRETRAIN, learning_rate=LR_PRETRAIN, model_dir=str(model_dir_stage1), phases=tuple(p for p in ('train','val') if p in dataloaders))
        freeze_module_params(feature_mapper)
        if USE_QUANTUM:
            from src.model_builder import build_quantum_head
            head_stage2 = build_quantum_head(N_QUBITS, len(class_names), Q_DEPTH)
        else:
            head_stage2 = nn.Sequential(nn.Linear(N_QUBITS, 64), nn.ReLU(), nn.Linear(64, len(class_names)))
        stage2_model = nn.Sequential(feature_mapper, head_stage2).to(device)
        model_dir_stage2 = Path(config['model_dir']) / 'stage2'
        model_dir_stage2.mkdir(parents=True, exist_ok=True)
        stage2_model, history_stage2 = train_with_history(stage2_model, dataloaders, dataset_sizes, device=device, num_epochs=EPOCHS_FINETUNE, learning_rate=LR_FINETUNE, model_dir=str(model_dir_stage2), phases=tuple(p for p in ('train','val') if p in dataloaders))
        eval_model = stage2_model
        final_acc = history_stage2.get('val_acc', [0])[-1] if history_stage2.get('val_acc') else 0
else:
    model = build_model(config, class_names, dataloaders, device=device)
    model, history_stage1 = train_model(model, dataloaders, dataset_sizes, device, config['num_epochs'], config['learning_rate'], config['model_dir'])
    eval_model = model
    final_acc = history_stage1.get('test_acc', [0])[-1] if history_stage1.get('test_acc') else 0

print('Final val/test acc estimate:', final_acc)


In [ ]:
def plot_history(history, title, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(10,4))
    for phase, color in [('train','blue'), ('val','red'), ('test','green')]:
        if f'{phase}_loss' in history:
            axes[0].plot(history[f'{phase}_loss'], label=f'{phase}', color=color)
        if f'{phase}_acc' in history:
            axes[1].plot(history[f'{phase}_acc'], label=f'{phase}', color=color)
    axes[0].set_title(f'{title} - Loss'); axes[1].set_title(f'{title} - Accuracy')
    for ax in axes:
        ax.legend(); ax.grid(True)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print('Saved plot to', save_path)
    plt.show()

plot_history(history_stage1, 'Stage 1', save_path=Path(config['model_dir'])/'stage1_metrics.png')
if history_stage2:
    plot_history(history_stage2, 'Stage 2', save_path=Path(config['model_dir'])/'stage2_metrics.png')


In [ ]:
metrics = {}
if 'val' in dataloaders:
    metrics['val'] = evaluate_model(eval_model, dataloaders['val'], class_names, device, config['model_dir'], split_name='val')
if 'test' in dataloaders:
    metrics['test'] = evaluate_model(eval_model, dataloaders['test'], class_names, device, config['model_dir'], split_name='test')
print(metrics)


In [ ]:
if USE_QUANTUM:
    try:
        analyze_trained_quantum_head(eval_model, n_qubits=N_QUBITS, q_depth=Q_DEPTH, device=device, save_dir=config.get('model_dir'))
    except Exception as exc:
        print('Quantum recap failed:', exc)
else:
    print('Quantum recap skipped (USE_QUANTUM=False)')
